In [1]:
from __future__ import annotations

import argparse
import json
import os
import re
import unicodedata
from pathlib import Path
from typing import Any

import pytesseract
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pdf2image import convert_from_path
from PIL import ImageOps
from rapidfuzz import fuzz
from pypdf import PdfReader
from tqdm import tqdm


d:\AI-ML\AI Projects\Study RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
PROJECT_ROOT = Path.cwd().parent 

TOC_KEYWORDS = ("সূচিপত্র", "সুচিপত্র", "বিষয়সূচি")
DEFAULT_MATCH_THRESHOLD = 80
POOPLER_PATH = r"C:\poppler\Library\bin"


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    is_separator_regex=False,
    separators=["\n\n\n", "\n\n", "\n", "।", " ", ""],
)



In [3]:
with open("D:\AI-ML\AI Projects\Study RAG\chapters.json", "r", encoding="utf-8") as file:
        INDEX_INFOS = json.load(file)

<>:1: SyntaxWarning: invalid escape sequence '\A'
<>:1: SyntaxWarning: invalid escape sequence '\A'
C:\Users\Tanvir\AppData\Local\Temp\ipykernel_26220\2489850357.py:1: SyntaxWarning: invalid escape sequence '\A'
  with open("D:\AI-ML\AI Projects\Study RAG\chapters.json", "r", encoding="utf-8") as file:


In [4]:
INDEX_INFOS

{'all_books': {'class_5': {'bangla': [{'chapter_number': 1,
     'chapter_name': 'বৈচিত্র্যময় বাংলাদেশ',
     'starting_page': 1},
    {'chapter_number': 2, 'chapter_name': 'তিতুমীর', 'starting_page': 6},
    {'chapter_number': 3, 'chapter_name': 'দূরের পাল্লা', 'starting_page': 19},
    {'chapter_number': 4, 'chapter_name': 'পত্র লিখি', 'starting_page': 24},
    {'chapter_number': 5, 'chapter_name': 'ঠিক আছে', 'starting_page': 28},
    {'chapter_number': 6, 'chapter_name': 'সুখ আর দুখু', 'starting_page': 32},
    {'chapter_number': 7, 'chapter_name': 'সাইক্লোন', 'starting_page': 38},
    {'chapter_number': 8,
     'chapter_name': 'রয়েল বেঙ্গল টাইগার',
     'starting_page': 42},
    {'chapter_number': 9,
     'chapter_name': 'টুকটুক ও চিকু',
     'starting_page': 47},
    {'chapter_number': 10, 'chapter_name': 'রাখাল ছেলে', 'starting_page': 53},
    {'chapter_number': 11, 'chapter_name': 'কুটির শিল্প', 'starting_page': 57},
    {'chapter_number': 12,
     'chapter_name': 'শিষ্যের সাধনা

In [5]:

def load_chapters(chapters_path: Path, class_no: int, subject_name: str) -> list[dict[str, Any]]:
    with chapters_path.open("r", encoding="utf-8") as file:
        index = json.load(file)

    class_key = f"class_{class_no}"
    books = index["all_books"][class_key]
    subject_key = subject_name.casefold()
    if subject_key == "math":
        subject_key = "mathematics"
    if subject_key not in books:
        available = ", ".join(books)
        raise ValueError(f"No index for {class_key}/{subject_name}. Available: {available}")

    chapters = books[subject_key]
    if not chapters:
        raise ValueError(f"No chapters found for {class_key}/{subject_key}")
    return sorted(chapters, key=lambda chapter: chapter["chapter_number"])


In [ ]:


def find_pdf_path(project_root: Path, class_no: int, subject_name: str) -> Path:
    class_pdf_dir = project_root / "pdf" / f"Class_{class_no}"
    subject_stem = "Math" if subject_name.casefold() in {"math", "mathematics"} else subject_name
    pdf_path = next(
        (
            candidate
            for candidate in class_pdf_dir.glob("*.pdf")
            if candidate.stem.casefold() == subject_stem.casefold()
        ),
        class_pdf_dir / f"{subject_stem}.pdf",
    )
    if not pdf_path.is_file():
        raise FileNotFoundError(f"PDF not found for class {class_no}, subject {subject_name}: {pdf_path}")
    return pdf_path


In [ ]:


def ocr_pdf_to_pages(pdf_path: Path, poppler_path: str | None = None, lang: str = "ben", dpi: int = 300) -> list[dict[str, Any]]:
    
    convert_options: dict[str, Any] = {"dpi": dpi}
    if poppler_path:
        convert_options["poppler_path"] = poppler_path

    page_count = len(PdfReader(str(pdf_path)).pages)
    pages = []
    batch_size = 8
    with tqdm(total=page_count, desc="OCR processing") as progress:
        for first_page in range(1, page_count + 1, batch_size):
            last_page = min(first_page + batch_size - 1, page_count)
            images = convert_from_path(
                str(pdf_path),
                first_page=first_page,
                last_page=last_page,
                **convert_options,
            )
            for page_index, image in enumerate(images, start=first_page):
                image = ImageOps.autocontrast(image.convert("L"))
                text = pytesseract.image_to_string(image, lang=lang)
                pages.append({"pdf_page_index": page_index, "text": text})
                progress.update(1)
            images.clear()
    return pages



In [ ]:
metas = {}
for i in INDEX_INFOS['all_books']['class_5']['english']:
    # print(i)

    if i.get("starting_page") == 57:
        if i.get("sub_chapters"):
            metas["part"] = i.get("chapter_name")
            for sub_chapter in i.get("sub_chapters"):

                if sub_chapter.get("starting_page") == 57:

                    metas["chapter"] = sub_chapter.get("sub_chapter_name")
                    if sub_chapter.get("writer_name"):
                        metas["writer_name"] = sub_chapter.get("writer_name")

        else:
            metas["chapter"] = i.get("chapter_name")

        
print(metas)
                

{'chapter': 'কুটির শিল্প'}


## self written

In [49]:

def ocr_pdf_to_pages(pdf_path : Path, poppler_path, lang='ben'):

    class_number = pdf_path.parent.name.lower()
    sub_name = pdf_path.stem.lower()

    keywords=("সূচিপত্র", "সুচিপত্র", "বিষয়সূচি", "CONTENTS", "contents")
    tracker = False
    page_num = 0

    page_count = len(PdfReader(str(pdf_path)).pages)
    images = convert_from_path(pdf_path, dpi=400, poppler_path=poppler_path)
    pages = []

    for i, img in enumerate(images, start=1):
        img = ImageOps.autocontrast(img.convert('L'))
        text = pytesseract.image_to_string(img, lang=lang)

        if any(k in text for k in keywords):
            tracker = True

        if tracker:
            pages.append({"page_number": page_num, "page_content": text, "source": sub_name, "class_number": class_number})
            page_num += 1
        else:
            pages.append({"page_number": -1, "page_content": text, "source": sub_name, "class_number": class_number})

    return pages

In [ ]:
from pathlib import Path
book_dicts = ocr_pdf_to_pages(Path(r"D:\AI-ML\AI Projects\Study RAG\pdf\Class_6\BGS.pdf"), POOPLER_PATH)

# 2. Convert to LangChain Documents
documents = []

for item in book_dicts:
    # Extract the main text string (defaulting to empty string if missing)
    content = item.pop("page_content", "") 

    chapter_name = None  # default if no chapter matches
    for i in INDEX_INFOS['all_books'][item.get("class_number")][item.get("source")]:
        if i.get("starting_page") == item.get("page_number"):
            chapter_name = i.get("chapter_name")
    item["chapter_name"] = chapter_name
    
    # Everything else remaining in the dict automatically becomes metadata
    doc = Document(page_content=content, metadata=item)
    documents.append(doc)

documents


In [ ]:
p = Path(r"D:\AI-ML\AI Projects\Study RAG\pdf\Class_7\English_Grammar.pdf")


print(folder, filename)

class_7 english_grammar


In [ ]:

def load_or_ocr_pages(
                    pdf_path: Path,
                    cache_path: Path,
                    poppler_path: str | None = None,
                    lang: str = "ben",
                    force_ocr: bool = False,
                    ) -> list[dict[str, Any]]:
    
    stat = pdf_path.stat()
    signature = {"size": stat.st_size, "modified_ns": stat.st_mtime_ns}
    if cache_path.exists() and not force_ocr:
        with cache_path.open("r", encoding="utf-8") as file:
            cache = json.load(file)
        if cache.get("source_pdf") == str(pdf_path.resolve()) and cache.get("signature") == signature:
            print(f"Using cached page OCR: {cache_path}")
            return cache["pages"]

    pages = ocr_pdf_to_pages(pdf_path, poppler_path=poppler_path, lang=lang)
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    with cache_path.open("w", encoding="utf-8") as file:
        json.dump(
            {"source_pdf": str(pdf_path.resolve()), "signature": signature, "pages": pages},
            file,
            ensure_ascii=False,
        )
    return pages